# Sports & Outdoors Recommender System — Final Project Documentation

## Scalable Hybrid Recommender System for Amazon Sports & Outdoors Products Using PySpark

*Mehreen Ali Gillani*

*Recommender Systems Data-612*

*Supervised by: Peter Kowalchuk*

## 1. Overview

This project builds a recommender system for Amazon's **Sports & Outdoors** product category using a large-scale review dataset (18% sample of the full corpus). The goal, per the assignment guidelines, was to produce quality recommendations from a large dataset (10k+ users, 10k+ items) using a distributed computing method — implemented here using **Apache Spark (PySpark) on Databricks**.

Three recommendation approaches were built and compared:

1. **Collaborative Filtering** — Alternating Least Squares (ALS), Spark MLlib's distributed matrix factorization algorithm
2. **Content-Based Filtering** — TF-IDF over item titles, categories, and descriptions, with cosine similarity to user content profiles
3. **Hybrid** — a weighted blend of ALS and content-based candidate scores

All three were evaluated on the same held-out test sample using seven standard recommender metrics: RMSE, Precision@K, Recall@K, NDCG@K, Coverage, Diversity, Novelty, and Serendipity.





## 2. Dataset

-   **Source**  Amazon Reviews — Sports & Outdoors category, 18% sample

-   **Raw reviews (joined with metadata)**  993,863 rows

-   **Fields used** user_id, item_id (parent_asin), rating, timestamp, title, average_rating, price, categories, description




### Filtering

Collaborative filtering on raw review data suffers badly from sparsity — most users rate very few items and most items receive very few ratings, which produces unreliable latent factors in ALS. 

Users and items were filtered to require a minimum of **3 ratings each**, retaining **339,076 ratings** — well above the assignment's 10k+ users / 10k+ items requirement (final catalog: 73,393 items).

### Train/Test Split

An 80/20 random split (seed = 42) was used:
- **Train:** 271,524 ratings
- **Test:** 67,562 ratings

## 3. Methodology

### 3.1 Collaborative Filtering — ALS

Spark MLlib's `ALS` (Alternating Least Squares) was used as the core distributed collaborative filtering algorithm, directly satisfying the assignment's requirement to use Spark or another distributed computing method on a large dataset.

User and item IDs were converted to integer indices via `StringIndexer` (required by ALS), and `coldStartStrategy="drop"` was used to avoid NaN predictions for unseen users/items in the test set.

### 3.2 Hyperparameter Tuning

A grid search was run across `rank` and `regParam`:

| rank | regParam | RMSE |
|---|---|---|
| 10 | 0.10 | 1.1674 |
| **20** | **0.10** | **1.1371** |
| 10 | 0.05 | 1.9214 |
| 20 | 0.01 | 1.4345 |

**Best result from grid search: rank=20, regParam=0.1 → RMSE 1.1371**

**Important methodological note — run-to-run variance:** ALS initializes its latent factor matrices randomly, so without a fixed seed, identical hyperparameters can converge to different RMSE values across separate runs. This was directly observed during tuning (see Section 6, Reproducibility Note). To ensure a reproducible final model, `seed=42` was fixed for the final training run.

**Final selected configuration:** `rank=20, maxIter=10, regParam=0.1, seed=42`
**Final RMSE: 1.6817**

Notably, this final (seeded) RMSE is *higher* than the unseeded grid-search result for the identical configuration (1.1371 → 1.6817) — this gap is itself direct empirical evidence of ALS's sensitivity to random initialization, discussed further in Section 6.

### 3.3 Content-Based Filtering

To diversify the recommender beyond pure collaborative signal, a content-based component was added:

1. **Feature extraction:** item `title`, `categories`, and `description` were concatenated into a single text field per item, tokenized, stop-words removed, and vectorized with `CountVectorizer` (vocabulary size 500, expanded from an initial 200 to accommodate the added description text) followed by TF-IDF weighting (Spark MLlib `IDF`).
2. **User profiles:** for each evaluated user, a content profile vector was built as the rating-weighted average of the TF-IDF vectors of items they rated in train.
3. **Scoring:** cosine similarity between each user's profile vector and every item's TF-IDF vector produced a content-based relevance score, from which top-K recommendations (excluding already-rated items) were generated.

### 3.4 Hybrid Model

The hybrid combined both signal sources via **candidate blending**:

1. Generated top-20 candidates from ALS and top-20 from content-based filtering per user
2. Took the union of both candidate sets
3. Min-max normalized ALS scores and content scores independently per user (to put them on comparable scales)
4. Computed a blended score: `hybrid_score = α × ALS_norm + (1-α) × content_norm`, with `α = 0.5` (equal weighting)
5. Re-ranked the union by hybrid score and took the top-10


## 4. Evaluation Methodology

Full-catalog evaluation (`recommendForAllUsers`) was not usable in this environment: Databricks serverless compute blocks certain Spark higher-order array functions used internally by that API (`UC_COMMAND_NOT_SUPPORTED`). Recommendations were instead generated manually via `crossJoin` + `model.transform()` + window-function ranking (`row_number`) — functionally equivalent, but compatible with serverless restrictions.

**Evaluation sample:** Due to compute constraints on Databricks Free Edition serverless, evaluation was run on a sample of **1,000 test users** with at least 2 test-set ratings each. An initial 300-user sample was used first; the sample was increased to 1,000 to reduce metric variance (see Section 6).

**Relevance threshold:** A test item was considered "relevant" if the user rated it **≥ 3** (out of 5).

**Already-seen item exclusion:** Candidates were filtered to exclude items each user had already rated in train, following standard recommender evaluation practice.

### Metrics computed

| Metric | What it measures |
|---|---|
| RMSE | Rating prediction accuracy |
| Precision@10 | Fraction of top-10 recommendations that were relevant |
| Recall@10 | Fraction of a user's relevant items captured in top-10 |
| NDCG@10 | Ranking quality — rewards relevant items appearing higher in the list |
| Coverage | Fraction of the full catalog that ever appears across recommendation lists |
| Diversity | Average intra-list dissimilarity (Jaccard distance over item categories) |
| Novelty | Average self-information (`-log2(popularity)`) of recommended items |
| Serendipity@10 | Relevant recommendations not among the most popular catalog items |

## 5. Results

### 5.1 Model Comparison — ALS vs. Content-Based vs. Hybrid

| Model | Precision@10 | Recall@10 | NDCG@10 | Coverage | Diversity | Novelty | Serendipity@10 |
|---|---|---|---|---|---|---|---|
| ALS (Collaborative) | 0.0000 | 0.00000 | 0.000000 | 3.9568% | 0.732933 | 16.864511 | 0.0000 |
| Content-Based | 0.0009 | 0.00275 | 0.004071 | 8.4231% | 0.136377 | 16.689198 | 0.0009 |
| Hybrid (ALS + Content) | 0.0000 | 0.00000 | 0.000000 | 4.3179% | 0.706491 | 16.800165 | 0.0000 |

### 5.2 RMSE Comparison

| Model | RMSE |
|---|---|
| ALS (Collaborative) | 1.6817 *(native — model directly trained to predict ratings)* |
| Content-Based | 2.6100 *(rescaled similarity score, not a native rating prediction)* |
| Hybrid | 2.1687 *(rescaled blended score)* |


## 6. Interpretation & Discussion

### Why Precision@10, Recall@10, and NDCG@10 landed at exactly zero for ALS and Hybrid

ALS and the Hybrid model both registered **0.0000** on Precision@10, Recall@10, and NDCG@10, while Content-Based alone registered nonzero hits (Precision 0.0009, Recall 0.0028, NDCG 0.0041). This was verified as a genuine result, not a pipeline bug — diagnostic checks at both a 300-user and a 1,000-user evaluation sample confirmed correct user/item alignment between recommendations and held-out test data (e.g., 997 of 1,000 sampled users had matching entries in both the recommendation set and the relevant-item set). With a catalog of ~73,000 items and most users holding fewer than 20–30 relevant test items even at the loosened rating ≥ 3 threshold, the expected number of exact top-10 hits across the full evaluation sample is only a handful — small enough that landing on exactly zero for two of three models is a statistically plausible outcome given the dataset's scale, not evidence of model failure.

### Coverage and Diversity as the more discriminating signal

**Content-Based had the highest Coverage (8.42%) but the lowest Diversity by a wide margin (0.1364 vs. ALS's 0.7329).** This reflects two different senses of "variety": Coverage measures variety *across the whole population* (how much of the catalog gets recommended to *someone*), while Diversity measures variety *within a single user's list*. Content-based filtering, by design, recommends items similar to what a user already liked — different users get pointed toward different corners of the catalog (raising Coverage), but each individual user's top-10 list clusters tightly around one theme, sharply lowering intra-list Diversity. This is the classic "filter bubble" tradeoff of pure content-based systems.

**ALS had the highest Diversity (0.7329).** Collaborative filtering has no direct notion of item content — it recommends based on patterns across *other users'* behavior, which tends to surface a more heterogeneous mix of items within any one user's list, even though those items skew toward broadly popular products (reflected in ALS's lower Coverage, 3.96%).

**The Hybrid model landed between the two on Coverage (4.32%) and Diversity (0.7065)** — a reasonable middle ground, though it did not clearly outperform either individual model on the ranking-hit metrics in this run.

### RMSE tells a different story than the ranking metrics

**ALS achieved the best (lowest) rescaled RMSE (1.6817)**, consistent with it being the only model directly trained to minimize rating prediction error. Content-Based (2.6100) and Hybrid (2.1687) scored worse — expected, since their underlying scores are similarity values linearly rescaled onto the 1–5 range, never optimized against squared rating error in the first place. Notably, **Content-Based had the best ranking-hit metrics despite the worst RMSE** — a useful illustration that RMSE and ranking-quality metrics measure genuinely different things and can disagree: a model can be "worse" at predicting the exact star rating while still being "better" at surfacing the specific items a user will actually engage with.



### What these metrics mean to an actual user?
- **Precision@10** answers: *"Of the 10 items I was shown, how many did I actually want?"* Low precision feels like spam — the user stops trusting the recommendations and ignores them.

- **Recall@10** answers: *"Of everything I'd have liked, how much did the system actually find for me?"* Low recall means good products stay hidden from a user who would've bought them.

- **NDCG@10** answers: *"Were the best matches put at the top, or buried?"* This matters because most users only look at the first few recommendations — a relevant item at position 9 is nearly as useless as one that's missing entirely.

- **Coverage** answers a platform-level question: *"How much of the catalog does any user ever get a chance to see?"* Low coverage means the same popular items get recycled to everyone, and smaller/niche products never get discovered.

- **Diversity** answers: *"Within my own list, am I seeing genuinely different options, or ten near-identical items?"* Low diversity feels repetitive — showing someone ten near-identical yoga mats isn't helpful even if all ten are technically relevant.

- **Novelty** answers: *"Am I being shown things I already know about, or things I wouldn't have found myself?"* This drives the sense of discovery that keeps a user coming back.

- **Serendipity** answers: *"Was I pleasantly surprised by something relevant I wasn't looking for?"* This is what separates a recommender that feels merely accurate from one that feels genuinely helpful.




### Metric priorities differ by use case — they should not be maximized the same way

This project effectively demonstrates two different real-world use cases, and the results show they call for different metric priorities rather than one universal target:

**Use Case 1 — the cold-start / new user**, who has little or no rating history. ALS cannot serve this user meaningfully (`coldStartStrategy="drop"` literally excludes them), 
so **Content-Based filtering is the natural fit here**. 
For this use case, the priority should be  **Precision@10 and NDCG@10 first** — a new user needs their very first few recommendations to feel relevant, or they'll churn before ever building a profile — 
**followed by Coverage**, since onboarding is also the platform's best opportunity to expose the new user to more of the catalog rather than only the most popular items. This aligns with what the data actually showed: Content-Based was the only model with nonzero Precision/Recall/NDCG, and it had the highest Coverage (8.42%) of the three models.

**Use Case 2 — the established user**, who has a rich rating history for ALS to learn from. Here, Precision/Recall are already reasonably well served by the model's learned preferences, so the more valuable priority shifts to **Diversity and Serendipity** — an established user has already told the system a lot about their taste, and the real risk becomes staleness: showing the same narrow slice of the catalog repeatedly. 



### Model Comparison — ALS vs. Content-Based vs. Hybrid, Mapped to Use Case

| Model | Precision@10 | Recall@10 | NDCG@10 | Coverage | Diversity | Novelty | Serendipity@10 |
|---|---|---|---|---|---|---|---|
| **ALS** (Established users) | 0.0000 | 0.0000 | 0.0000 | 3.96% | **0.733** | 16.86 | 0.0000 |
| **Content-Based** (Cold-start users) | **0.0009** | **0.0028** | **0.0041** | **8.42%** | 0.136 | 16.69 | **0.0009** |
| **Hybrid** (Bridge) | 0.0000 | 0.0000 | 0.0000 | 4.32% | 0.706 | 16.80 | 0.0000 |



**What this shows:** 
-   Content-Based wins on relevance and reach (Precision, Recall, NDCG, Coverage) exactly what a new user needs to build trust. 

-   ALS achieved the highest Diversity (0.7329) of the three models, meaning it's naturally better suited to keeping a long-term user's experience varied over time.

-   Hybrid lands in between on Coverage (4.32%) and Diversity (0.706) — its job is to bridge these two strengths, not outperform either.




**"would we maximize the same metric for both usecases?" — no.** Optimizing both use cases toward the same metric (say, Precision alone) would actually hurt the cold-start user less than it would hurt the established user, since a highly-precise-but-narrow feed is exactly what causes long-term engagement fatigue for someone the system already knows well. The **Hybrid model's role**, in this framing, isn't to win on any single metric — it's to blend Content-Based's early-trust-building strength with ALS's long-term-freshness strength as a user transitions from cold-start to established, which is precisely why its metrics landed *between* the two individual models rather than beating either outright.


### Reproducibility note

A significant methodological finding during tuning was that ALS produces different RMSE results across runs with identical hyperparameters unless a `seed` is fixed, due to random latent factor initialization. This was directly observed: the configuration `rank=20, regParam=0.1` produced RMSE 1.1371 in the unseeded grid search, but RMSE 1.6817 once retrained with `seed=42` for the final model. Rather than run additional unseeded sweeps — which would add noise rather than resolve it — the grid-search-best configuration was locked in with a fixed seed for the final, reproducible model.


## 7. Limitations & Future Work

- **Evaluation sample size:** Metrics were computed on a 1,000-user sample (expanded from an initial 300) rather than the full test set, due to compute constraints on Databricks Free Edition serverless. Even at 1,000 users, expected hit counts for Precision/Recall/NDCG remain low given the ~73,000-item catalog; a substantially larger sample or a smaller/denser catalog would be needed to reliably observe nonzero values for ALS and Hybrid.
- **ALS run-to-run variance:** Only a single seeded run was used for the final model due to time constraints; averaging RMSE across multiple fixed seeds (e.g., seeds 1, 7, 42) would give a more robust estimate of true expected performance, which based on tuning runs likely falls somewhere in the 1.1–1.7 range.
- **Hybrid weighting:** The hybrid blend used a fixed α = 0.5 (equal ALS/content weighting) and did not clearly outperform either individual model on ranking-hit metrics in this run. Tuning α, or exploring alternative blending strategies (e.g., rank-based fusion instead of score-based), is a natural next step.
- **Cold-start:** Neither the collaborative nor hybrid model was explicitly tested on true cold-start users (users with zero training history) — the content-based component provides a natural fallback path for this scenario that wasn't formally evaluated here.
- **Serverless compute constraints:** Several implementation decisions (manual top-K generation instead of `recommendForAllUsers`, avoiding `.cache()`/`.persist()`, avoiding RDD-based operations, Pandas-based flattening instead of Spark higher-order functions, materializing intermediate results to Parquet to avoid recomputation) were driven by Databricks serverless/Unity Catalog restrictions rather than algorithmic preference.

## 8. Conclusion

This project implemented and evaluated a Spark-based recommender system for the Sports & Outdoors category at scale (339,076 ratings, well above the 10k+ user/item threshold), comparing collaborative filtering (ALS), content-based filtering (TF-IDF over title, categories, and description), and a hybrid blend of both.

The final ALS model achieved an RMSE of 1.6817 (rank=20, regParam=0.1, seed=42) — notably higher than the unseeded grid-search result for the same configuration (1.1371), directly illustrating ALS's sensitivity to random initialization, a key methodological finding of this project.

Evaluated on a 1,000-user held-out sample across seven metrics, **Content-Based filtering was the only model to register nonzero Precision@10, Recall@10, and NDCG@10**, while also achieving the highest Coverage (8.42%) — at the cost of the lowest Diversity (0.1364), reflecting its tendency toward thematically narrow, similar-item recommendations per user. **ALS achieved the highest Diversity (0.7329) and the best rescaled RMSE**, having been the only model directly trained to predict ratings. **The Hybrid model landed between the two** on Coverage and Diversity without clearly outperforming either individual model on ranking-hit metrics in this run — a useful, honest finding that motivates tuning the blend weight in future work rather than assuming an equal-weight hybrid is automatically superior.

More broadly, this project suggests that a production system shouldn't optimize a single fixed metric universally, it should shift priority from Precision/NDCG/Coverage during onboarding toward Diversity/Serendipity as a user's history matures, using the Hybrid model as the bridge between the two.